# Spiking Neural Networks with sinabs
### A hands-on tutorial: from neuron to trained model

**Prerequisites:** Python, basic PyTorch (`nn.Module`, training loop). No neuroscience background needed.

**What you will build:** A spiking neural network trained on N-MNIST — the neuromorphic version of MNIST — using the [sinabs](https://sinabs.readthedocs.io) library.

**What you will understand:**
- Why neurons spike instead of outputting continuous values
- How a spiking neuron integrates input over time
- How to train an SNN with backpropagation through time (BPTT)
- Why surrogate gradients are necessary and how they work

## Before you start

Activate the `sinabs_tutorial` conda environment as your kernel. See [README.md](README.md) for one-time setup instructions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyArrowPatch
import torch
import torch.nn as nn
import tonic
import tonic.transforms as transforms
import sinabs
import sinabs.layers as sl
from torch.utils.data import DataLoader

torch.manual_seed(42)
np.random.seed(42)
print(f"sinabs version: {sinabs.__version__}")
print(f"PyTorch version: {torch.__version__}")

---
## 1. Why Spikes?

A standard artificial neuron outputs a **continuous value** — a floating-point number passed through ReLU or sigmoid. Biological neurons communicate very differently: they either fire an electrical pulse (a **spike**, value = 1) or stay silent (value = 0). Nothing in between.

```
Standard neuron:   output = ReLU(w·x + b)     ← real number, every step
Spiking neuron:    output = 0 or 1             ← binary event, only when threshold crossed
```

**Why bother?**
1. **Energy efficiency** — a spike is transmitted only when something happens. Silent neurons cost almost nothing on neuromorphic hardware.
2. **Event-driven sensors** — event cameras (DVS) produce exactly this format: sparse binary events in time. SNNs process them natively.
3. **Temporal dynamics** — a spiking neuron has memory: it accumulates charge over time before firing. This is useful for processing temporal data.

The key challenge: if the neuron output is always 0 or 1, how do we train with backpropagation? We will answer this in Section 5.

---
## 2. Event-Based Data: N-MNIST

**N-MNIST** (Neuromorphic MNIST) was recorded by pointing a DVS event camera at an LCD screen displaying standard MNIST digits while moving the camera slightly. The result: each digit is represented as a stream of events in time instead of a static pixel image.

A **DVS camera** does not capture frames. Instead, each pixel fires independently whenever its brightness *changes*:
- **ON event** (polarity = 1): brightness increased
- **OFF event** (polarity = 0): brightness decreased

We accumulate events into fixed time windows to get discrete frames:

```
static image shape:      (C, H, W)              e.g. (1, 28, 28)
event-based SNN shape:   (T, C, H, W)            e.g. (T, 2, 34, 34)
                          ↑                              ↑
                     time slices              ON/OFF polarity channels
```

In [ ]:
NUM_TIMESTEPS = 10
SENSOR_SIZE = tonic.datasets.NMNIST.sensor_size  # (34, 34, 2)

frame_transform = transforms.Compose([
    transforms.Denoise(filter_time=10_000),
    transforms.ToFrame(sensor_size=SENSOR_SIZE, n_time_bins=NUM_TIMESTEPS),
])

dataset_train_raw = tonic.datasets.NMNIST(save_to="./data", train=True,  transform=frame_transform)
dataset_test_raw  = tonic.datasets.NMNIST(save_to="./data", train=False, transform=frame_transform)

# Cache preprocessed frames to disk — slow only on the very first run, fast every run after
dataset_train = tonic.DiskCachedDataset(dataset_train_raw, cache_path="./data/cache/nmnist_train")
dataset_test  = tonic.DiskCachedDataset(dataset_test_raw,  cache_path="./data/cache/nmnist_test")

print(f"Training samples: {len(dataset_train)}")
print(f"Test samples:     {len(dataset_test)}")

sample, label = dataset_train[0]
print(f"\nOne sample shape: {sample.shape}   → (T, C, H, W)")
print(f"Label: {label}")
print("\nNote: first run builds the cache (a few minutes). All runs after are fast.")

In [ ]:
sample_np = sample  # (T, 2, 34, 34)

T_show = 5
fig, axes = plt.subplots(2, T_show, figsize=(12, 5))
fig.suptitle(f'N-MNIST sample — label: {label}', fontsize=13, fontweight='bold')

row_labels = ['ON events', 'OFF events']
cmaps      = ['Reds',      'Blues']

for t in range(T_show):
    for ch in range(2):
        ax = axes[ch, t]
        ax.imshow(sample_np[t, ch], cmap=cmaps[ch], vmin=0, vmax=1)
        ax.set_title(f't = {t}', fontsize=10)
        ax.axis('off')

# Label each row in the leftmost column.
# ax.text with transform=ax.transAxes works even when axis('off') is set.
for ch in range(2):
    axes[ch, 0].text(-0.15, 0.5, row_labels[ch], transform=axes[ch, 0].transAxes,
                     fontsize=10, va='center', ha='right', fontweight='bold', rotation=90)

plt.tight_layout()
plt.show()
print("Most pixels are zero (silent) — events are sparse.")

---
## 3. The Integrate-and-Fire (IAF) Neuron

The IAF neuron is the simplest spiking neuron model. It has one internal variable: the **membrane voltage** `Vmem`.

At each timestep:

```
① Integrate:   Vmem  ←  Vmem + input
② Fire:        spike =  floor(Vmem / threshold)   (0 if below; 1, 2, … if above)
③ Reset:       Vmem  ←  Vmem − spike × threshold
               (subtracts once per spike — leftover charge is preserved)
```

Notice: `Vmem` carries over from one timestep to the next. A weak input that cannot cross the threshold at t=0 may still cause a spike at t=3 after accumulating charge. **The neuron has memory.**

**sinabs default (`MultiSpike`):** a neuron that accumulates Vmem = 2.3 with threshold = 1.0 fires twice in one step and resets to 0.3. With normalised inputs, Vmem rarely exceeds 2 × threshold, so you will almost always see at most one spike per step.

In [ ]:
def iaf_neuron_sim(inputs, threshold=1.0):
    """Simulate a single IAF neuron matching sinabs defaults:
    MultiSpike (floor(Vmem/threshold) spikes per step) + MembraneSubtract reset."""
    T = len(inputs)
    vmem = np.zeros(T + 1)   # vmem[t] = voltage before timestep t
    spikes = np.zeros(T)

    for t in range(T):
        vmem[t + 1] = vmem[t] + inputs[t]                  # ① integrate
        if vmem[t + 1] >= threshold:                        # ② fire
            spikes[t] = np.floor(vmem[t + 1] / threshold)  # MultiSpike: fire as many times as possible
            vmem[t + 1] -= spikes[t] * threshold            # ③ reset: subtract once per spike

    return vmem[1:], spikes   # return vmem at end of each step


# Two scenarios: strong input fires early; weak input fires late
T = 12
inputs_strong = np.array([0.4, 0.35, 0.3, 0.2, 0.15, 0.2, 0.3, 0.35, 0.25, 0.1, 0.2, 0.3])
inputs_weak   = np.array([0.1, 0.12, 0.09, 0.11, 0.1, 0.13, 0.1, 0.12, 0.09, 0.11, 0.1, 0.13])

vmem_strong, spikes_strong = iaf_neuron_sim(inputs_strong)
vmem_weak,   spikes_weak   = iaf_neuron_sim(inputs_weak)

timesteps = np.arange(T)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 7), sharey='row')
fig.suptitle('IAF Neuron: membrane voltage and spiking over time', fontsize=14, fontweight='bold')
threshold = 1.0

for col, (vmem, spikes, inp, title) in enumerate(zip(
    [vmem_strong, vmem_weak], [spikes_strong, spikes_weak],
    [inputs_strong, inputs_weak],
    ['Strong input  (fires often)', 'Weak input  (fires rarely)'],
)):
    ax0 = axes[0, col]
    ax0.bar(timesteps, inp, color='steelblue', alpha=0.8, width=0.6)
    ax0.set_title(title, fontsize=12); ax0.set_ylabel('Input current', fontsize=11)
    ax0.set_xlabel('Timestep', fontsize=10); ax0.set_xticks(timesteps)
    ax0.set_ylim(0, 0.6); ax0.grid(axis='y', alpha=0.3)

    ax1 = axes[1, col]
    spike_times = np.where(spikes > 0)[0]
    vmem_display = vmem.copy()
    for t in spike_times:
        vmem_display[t] = vmem[t] + spikes[t] * threshold

    ax1.plot(timesteps, vmem_display, color='darkorange', lw=2, marker='o', ms=5, label='Vmem', zorder=3)
    ax1.axhline(threshold, color='red', ls='--', lw=1.5, label=f'Threshold = {threshold}')

    for t in spike_times:
        pre, post, n = vmem[t] + spikes[t] * threshold, vmem[t], int(spikes[t])
        ax1.annotate('', xy=(t, post), xytext=(t, pre),
                     arrowprops=dict(arrowstyle='->', color='purple', lw=1.5, connectionstyle='arc3,rad=0.4'))
        ax1.text(t + 0.15, (pre + post) / 2, 'reset', color='purple', fontsize=8, va='center')
        ax1.annotate('', xy=(t, pre + 0.25), xytext=(t, pre),
                     arrowprops=dict(arrowstyle='->', color='red', lw=2))
        ax1.text(t, pre + 0.32, 'spike' if n == 1 else f'{n}× spike',
                 ha='center', fontsize=9, color='red', fontweight='bold')

    ax1.set_ylabel('Membrane voltage (Vmem)', fontsize=11); ax1.set_xlabel('Timestep', fontsize=10)
    ax1.set_xticks(timesteps); ax1.set_ylim(-0.1, 1.9)
    ax1.legend(fontsize=10, loc='upper right'); ax1.grid(axis='y', alpha=0.3)

plt.tight_layout(); plt.show()

**What to notice:**
- The strong neuron crosses threshold quickly and fires multiple times, resetting each time
- The weak neuron slowly accumulates charge — it fires only after several timesteps of integration
- The purple arrow shows the **reset**: after a spike, Vmem drops by exactly `threshold`, not to zero

This is how an SNN encodes information: not as a real number, but as the **timing and rate of spikes**.

---
## 4. From Single Neuron to a Network: sinabs and IAFSqueeze

### The batching challenge

Standard `nn.Conv2d` accepts 4D tensors: `(Batch, Channels, H, W)`. It has no concept of time. But our data has shape `(Batch, T, C, H, W)`.

sinabs solves this with the **squeeze trick**: merge `Batch` and `T` into one dimension before the Conv layers, then split them back inside the spiking layer.

```
Input:          (B,  T, C, H, W)     e.g. (32, 10, 2, 34, 34)
                  ↓  squeeze: B × T
To Conv2d:      (320, 2, 34, 34)     ← treated as a big batch of 320 images
                  ↓  Conv2d (stateless)
                (320, N, 34, 34)
                  ↓  IAFSqueeze (knows B=32, T=10)
                      → internally loops over 10 timesteps
                      → maintains Vmem for each of the 32 real samples
                (320, N, 34, 34)     ← binary spikes, reshaped back
```

`IAFSqueeze` is constructed with `batch_size` and `num_timesteps` so it knows how to interpret the merged dimension.

In [ ]:
BATCH_SIZE = 32

# Reshape input: (B, T, C, H, W) → (B*T, C, H, W)
# sinabs handles this automatically; here we do it manually to see the shapes

x_demo = torch.zeros(BATCH_SIZE, NUM_TIMESTEPS, 2, 34, 34)
print(f"Original input:     {tuple(x_demo.shape)}   (B, T, C, H, W)")

x_squeezed = x_demo.reshape(BATCH_SIZE * NUM_TIMESTEPS, 2, 34, 34)
print(f"After squeeze:      {tuple(x_squeezed.shape)}   (B×T, C, H, W)")

conv = nn.Conv2d(2, 8, kernel_size=3, padding=1)
x_conv_out = conv(x_squeezed)
print(f"After Conv2d:       {tuple(x_conv_out.shape)}   (B×T, filters, H, W)")

### Building the SNN model

Our architecture: two convolutional blocks followed by a fully connected output layer. Each Conv layer is followed by an `IAFSqueeze` spiking layer.

```
(B*T, 2, 34, 34)
    → Conv2d(2→16)  → IAFSqueeze
    → AvgPool2d(2)  
    → Conv2d(16→32) → IAFSqueeze
    → AvgPool2d(2)
    → Flatten
    → Linear(32*8*8→10) → IAFSqueeze   ← 10 classes
```

In [ ]:
def build_snn(batch_size, num_timesteps):
    return nn.Sequential(
        # Block 1
        nn.Conv2d(2, 16, kernel_size=3, padding=1, bias=False),
        sl.IAFSqueeze(batch_size=batch_size, num_timesteps=num_timesteps),
        nn.AvgPool2d(2),       # 34 → 17

        # Block 2
        nn.Conv2d(16, 32, kernel_size=3, padding=1, bias=False),
        sl.IAFSqueeze(batch_size=batch_size, num_timesteps=num_timesteps),
        nn.AvgPool2d(2),       # 17 → 8

        # Classifier
        nn.Flatten(),
        nn.Linear(32 * 8 * 8, 10, bias=False),
        sl.IAFSqueeze(batch_size=batch_size, num_timesteps=num_timesteps),
    )

model = build_snn(BATCH_SIZE, NUM_TIMESTEPS)
print(model)

# Verify shapes with a dummy forward pass
dummy = torch.zeros(BATCH_SIZE * NUM_TIMESTEPS, 2, 34, 34)
with torch.no_grad():
    out = model(dummy)
print(f"\nOutput shape: {tuple(out.shape)}   (B×T, num_classes)")

---
## 5. BPTT and the Surrogate Gradient

### The problem: dead gradients

Training needs gradients. The gradient of the spike function is:

```
spike = 1  if  Vmem ≥ threshold,  else 0

d(spike)/d(Vmem) = 0   almost everywhere   ← step function has zero slope
```

Zero gradient → weights never update → network cannot learn. This is the core challenge of SNN training.

### The fix: surrogate gradient

During the **backward pass only**, replace the zero slope with a smooth approximation peaked at the threshold. The **forward pass still produces true binary spikes** — only the gradient computation is approximated.

```
Forward:   spike = step(Vmem - threshold)        ← true binary (0 or 1)
Backward:  d(spike)/d(Vmem) ≈ smooth_fn(Vmem)   ← surrogate (non-zero near threshold)
```

In [ ]:
vmem_vals = np.linspace(-1.5, 2.5, 500)
threshold = 1.0
v = vmem_vals - threshold

step   = (vmem_vals >= threshold).astype(float)
sg_exp = np.exp(-4.0 * np.abs(v))
sg_gau = np.exp(-0.5 * (v / 0.4) ** 2); sg_gau /= sg_gau.max()
sg_mul = (0.5  * np.exp(-0.5 * (v / 0.3) ** 2)
          - 0.25 * (np.exp(-0.5 * ((v + 0.5) / 0.3) ** 2)
                  + np.exp(-0.5 * ((v - 0.5) / 0.3) ** 2)))

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle('Surrogate gradients: forward spike (dashed) vs backward gradient (colour)',
             fontsize=13, fontweight='bold')

for ax, name, sg, color in zip(
    axes,
    ['SingleExponential\n(sinabs default)', 'Gaussian', 'MultiGaussian\n(negative lobes)'],
    [sg_exp, sg_gau, sg_mul],
    ['steelblue', 'darkorange', 'forestgreen'],
):
    ax2 = ax.twinx()
    ax.plot(vmem_vals, step, 'k--', lw=2, label='Step (forward)')
    ax.axvline(threshold, color='red', lw=1, ls=':', alpha=0.6)
    ax.set_ylim(-0.2, 1.4); ax.set_ylabel('spike (0 or 1)', fontsize=10)
    ax2.plot(vmem_vals, sg, color=color, lw=2.5, label='Surrogate grad')
    ax2.axhline(0, color='gray', lw=0.8, alpha=0.4)
    ax2.set_ylabel('gradient magnitude', fontsize=10, color=color)
    ax2.tick_params(axis='y', labelcolor=color)
    ax.set_title(name, fontsize=12, fontweight='bold'); ax.set_xlabel('Vmem', fontsize=11)
    ax.text(threshold + 0.05, 0.05, 'θ', color='red', fontsize=12)
    h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, fontsize=9, loc='upper left')

plt.tight_layout(); plt.show()

**What each surrogate does:**
- **SingleExponential**: sharp peak at threshold, decays fast. Neurons far from threshold receive no gradient signal. Default in sinabs.
- **Gaussian**: symmetric bell curve. Slightly wider influence — neurons a bit further from threshold still receive some gradient.
- **MultiGaussian**: positive peak plus *negative lobes*. Neurons stuck just below threshold get a negative gradient, pushing them to either commit to firing or stay clearly silent. Produces more decisive networks.

### BPTT: backpropagation through time

During training, the forward pass unrolls the neuron across all T timesteps. The backward pass flows gradients back through the same unrolled graph — this is **Backpropagation Through Time (BPTT)**.

In [ ]:
T_bptt, threshold_val, beta_val = 6, 1.0, 4.0
toy_inputs = np.array([0.3, 0.25, 0.4, 0.2, 0.35, 0.3])

vmem_trace, spikes_trace = iaf_neuron_sim(toy_inputs, threshold=threshold_val)
vmem_display = vmem_trace.copy()
for t in np.where(spikes_trace > 0)[0]:
    vmem_display[t] = vmem_trace[t] + spikes_trace[t] * threshold_val
sg_vals = np.exp(-beta_val * np.abs(vmem_display - threshold_val))

fig = plt.figure(figsize=(16, 9))
gs = gridspec.GridSpec(3, 2, width_ratios=[1.6, 1.0], hspace=0.55, wspace=0.45)
ax_input, ax_vmem, ax_grad = [fig.add_subplot(gs[i, 0]) for i in range(3)]
ax_sg = fig.add_subplot(gs[:, 1])
fig.suptitle('BPTT in an SNN: forward pass and surrogate gradient', fontsize=14, fontweight='bold')
ts = np.arange(T_bptt)

# ① Input
ax_input.bar(ts, toy_inputs, color='steelblue', alpha=0.8, width=0.5)
ax_input.set_title('① Forward pass — input current per timestep', fontsize=11)
ax_input.set_ylabel('Input current', fontsize=11); ax_input.set_ylim(0, 0.6)
ax_input.set_xticks(ts); ax_input.set_xticklabels([f't={t}' for t in ts], fontsize=10)
ax_input.grid(axis='y', alpha=0.3)

# ② Vmem trace with forward / backward arrows and reset drops
ax_vmem.plot(ts, vmem_display, color='darkorange', lw=2.5, marker='o', ms=7, zorder=3, label='Vmem')
ax_vmem.axhline(threshold_val, color='red', ls='--', lw=1.5, label=f'Threshold θ = {threshold_val}')
ap_fwd = dict(arrowstyle='->', color='darkorange', lw=1.2, alpha=0.5)
for t in range(T_bptt - 1):
    ax_vmem.annotate('', xy=(t+0.9, vmem_display[t+1]), xytext=(t+0.1, vmem_display[t]), arrowprops=ap_fwd)
for t in range(T_bptt - 1, 0, -1):
    ax_vmem.annotate('', xy=(t-0.9, vmem_display[t-1]+0.08), xytext=(t-0.1, vmem_display[t]+0.08),
                     arrowprops=dict(arrowstyle='->', color='purple', lw=1.5, alpha=float(0.2+0.7*sg_vals[t])))
for t in np.where(spikes_trace > 0)[0]:
    pre, post, n = vmem_trace[t]+spikes_trace[t]*threshold_val, vmem_trace[t], int(spikes_trace[t])
    ax_vmem.annotate('', xy=(t, post), xytext=(t, pre),
                     arrowprops=dict(arrowstyle='->', color='purple', lw=1.5, connectionstyle='arc3,rad=0.4'))
    ax_vmem.text(t+0.12, (pre+post)/2, 'reset', color='purple', fontsize=8, va='center')
    ax_vmem.text(t, pre+0.1, 'spike!' if n == 1 else f'{n}× spike!',
                 ha='center', color='red', fontsize=9, fontweight='bold')
ax_vmem.legend(handles=[mpatches.Patch(color=c, label=l) for c, l in [
    ('darkorange', 'Forward (Vmem chain)'), ('purple', '← Backward (BPTT)'), ('red', 'Threshold / spike')]],
    fontsize=9, loc='upper left')
ax_vmem.set_title('② Membrane voltage — forward (orange) and backward gradient flow (purple)', fontsize=11)
ax_vmem.set_ylabel('Vmem', fontsize=11); ax_vmem.set_ylim(-0.1, 1.7)
ax_vmem.set_xticks(ts); ax_vmem.set_xticklabels([f't={t}' for t in ts], fontsize=10)
ax_vmem.grid(axis='y', alpha=0.3)

# ③ Gradient bars
bar_colors = ['red' if s > 0 else 'steelblue' for s in spikes_trace]
ax_grad.bar(ts, sg_vals, color=bar_colors, alpha=0.8, width=0.5)
[ax_grad.text(t, v+0.03, f'{v:.2f}', ha='center', fontsize=9) for t, v in enumerate(sg_vals)]
ax_grad.set_title('③ Gradient signal per timestep — red = spike occurred', fontsize=11)
ax_grad.set_ylabel('Surrogate gradient\nmagnitude', fontsize=11); ax_grad.set_ylim(0, 1.1)
ax_grad.set_xticks(ts); ax_grad.set_xticklabels([f't={t}' for t in ts], fontsize=10)
ax_grad.grid(axis='y', alpha=0.3)

# ④ SG curve — each dot shows where that timestep's Vmem lands on the gradient curve
vmem_range = np.linspace(-0.1, 1.8, 400)
sg_curve = np.exp(-beta_val * np.abs(vmem_range - threshold_val))
ax_sg.plot(vmem_range, sg_curve, color='gray', lw=2.5, alpha=0.7, label='SG curve')
ax_sg.fill_between(vmem_range, sg_curve, alpha=0.07, color='gray')
ax_sg.axvline(threshold_val, color='red', ls='--', lw=1.5, alpha=0.7, label='θ')
cmap = plt.cm.viridis
for t in range(T_bptt):
    v, sg, spike = vmem_display[t], sg_vals[t], spikes_trace[t] > 0
    c = cmap(t / max(T_bptt - 1, 1))
    ax_sg.plot([v, v], [0, sg], color=c, ls=':', lw=1.2, alpha=0.7)
    ax_sg.scatter([v], [sg], color=c, s=250 if spike else 130, marker='*' if spike else 'o',
                  zorder=5, edgecolors='black', lw=0.8)
    ax_sg.text(v, sg+0.05, f't={t}' + (' ★' if spike else ''), fontsize=9,
               ha='center', color=c, fontweight='bold')
ax_sg.set_title("④ Where each timestep's Vmem\nfalls on the SG curve", fontsize=12)
ax_sg.set_xlabel('Vmem (pre-fire)', fontsize=11); ax_sg.set_ylabel('Surrogate gradient magnitude', fontsize=11)
ax_sg.set_xlim(-0.1, 1.8); ax_sg.set_ylim(-0.05, 1.25)
ax_sg.grid(alpha=0.3); ax_sg.legend(fontsize=9)

plt.tight_layout(); plt.show()
print("Dots near θ → peak of SG curve → strong gradient.  Dots far from θ → tail → near-zero gradient.")

**BPTT summary:**
1. **Forward pass**: input is processed timestep by timestep; Vmem evolves and spikes occur
2. **Loss**: spike counts are accumulated over all T timesteps and used as logits → cross-entropy loss
3. **Backward pass**: gradients flow backward through the entire unrolled graph (all T steps)
4. **At each spike point**: the surrogate gradient replaces the zero derivative of the step function
5. **Weight update**: optimizer updates Conv and Linear weights based on these gradients
6. **Reset Vmem**: membrane state is cleared before the next batch

The surrogate gradient is the trick that makes SNN training on standard GPU hardware possible.

---
## 6. Training Loop

The loss function deserves attention. After the full forward pass, we have output spike counts of shape `(B×T, num_classes)`. We need to **sum across time** to get one prediction per sample:

```
output spikes:   (B×T, 10)  — binary, one row per timestep per sample
                   ↓  reshape to (B, T, 10)
                   ↓  sum over T
spike counts:    (B, 10)    — total spikes per class per sample → logits
                   ↓  cross-entropy
loss:            scalar
```

In [ ]:
from tonic.collation import PadTensors

train_loader = DataLoader(
    dataset_train,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,          # ensures every batch has exactly BATCH_SIZE samples
    collate_fn=PadTensors(batch_first=True),
)
test_loader = DataLoader(
    dataset_test,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=True,
    collate_fn=PadTensors(batch_first=True),
)

# Confirm batch shape
x_batch, y_batch = next(iter(train_loader))
print(f"Batch shape: {tuple(x_batch.shape)}   (B, T, C, H, W)")
print(f"Labels:      {tuple(y_batch.shape)}")

In [ ]:
device = (torch.device("cuda") if torch.cuda.is_available()
          else torch.device("mps") if torch.backends.mps.is_available()
          else torch.device("cpu"))
print(f"Using device: {device}")

model = build_snn(BATCH_SIZE, NUM_TIMESTEPS).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()


def train_one_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.float().to(device), y.long().to(device)
        B, T, C, H, W = x.shape
        x = x.reshape(B * T, C, H, W)
        spikes_out = model(x)
        spike_counts = spikes_out.reshape(B, T, -1).sum(dim=1)
        loss = loss_fn(spike_counts, y)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        sinabs.reset_states(model)
        total_loss += loss.item()
        correct += (spike_counts.argmax(dim=1) == y).sum().item()
        total += B
    return total_loss / len(loader), correct / total


@torch.no_grad()
def evaluate(model, loader, loss_fn, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.float().to(device), y.long().to(device)
        B, T, C, H, W = x.shape
        x = x.reshape(B * T, C, H, W)
        spike_counts = model(x).reshape(B, T, -1).sum(dim=1)
        loss = loss_fn(spike_counts, y)
        sinabs.reset_states(model)
        total_loss += loss.item()
        correct += (spike_counts.argmax(dim=1) == y).sum().item()
        total += B
    return total_loss / len(loader), correct / total

In [ ]:
NUM_EPOCHS = 5

train_losses, train_accs = [], []
test_losses,  test_accs  = [], []

for epoch in range(1, NUM_EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, loss_fn, device)
    te_loss, te_acc = evaluate(model, test_loader, loss_fn, device)

    train_losses.append(tr_loss);  train_accs.append(tr_acc)
    test_losses.append(te_loss);   test_accs.append(te_acc)

    print(f"Epoch {epoch}/{NUM_EPOCHS}  "
          f"train loss: {tr_loss:.4f}  train acc: {tr_acc:.3f}  "
          f"test loss: {te_loss:.4f}  test acc: {te_acc:.3f}")

---
## 7. Results

In [ ]:
epochs = range(1, NUM_EPOCHS + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Training results — SNN on N-MNIST', fontsize=13, fontweight='bold')

ax1.plot(epochs, train_losses, 'o-', label='Train', color='steelblue')
ax1.plot(epochs, test_losses,  's--', label='Test',  color='darkorange')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('Cross-entropy loss')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(epochs, train_accs, 'o-', label='Train', color='steelblue')
ax2.plot(epochs, test_accs,  's--', label='Test',  color='darkorange')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.set_title('Classification accuracy')
ax2.set_ylim(0, 1); ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
model.eval()
x_vis, y_vis = next(iter(test_loader))
x_vis = x_vis.float().to(device)
B, T, C, H, W = x_vis.shape
with torch.no_grad():
    out = model(x_vis.reshape(B * T, C, H, W)).cpu().numpy()
sinabs.reset_states(model)
out_reshaped = out.reshape(B, T, 10)

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
fig.suptitle('Output spike raster — each row = one output class, each column = one timestep',
             fontsize=12, fontweight='bold')
for i, ax in enumerate(axes):
    ax.imshow(out_reshaped[i].T, aspect='auto', cmap='Blues', vmin=0, vmax=1, interpolation='nearest')
    ax.set_title(f'Label: {y_vis[i].item()}', fontsize=11)
    ax.set_xlabel('Timestep', fontsize=10); ax.set_ylabel('Class neuron', fontsize=10)
    ax.set_yticks(range(10))
plt.tight_layout(); plt.show()
print("Classification: the class whose output neuron fired the most times wins.")

---
## Summary

| Concept | Key point |
|---|---|
| Spiking neuron | Outputs 0 or 1; accumulates charge in Vmem; fires when Vmem ≥ threshold |
| Event-based data | Shape `(T, C, H, W)`; sparse; ON/OFF channels from DVS camera |
| IAFSqueeze | Merges B and T before Conv layers; loops over T inside; maintains Vmem per sample |
| Surrogate gradient | Replaces zero derivative of step function in backward pass; makes BPTT possible |
| Loss | Spike counts summed over all T timesteps → cross-entropy; every timestep contributes |
| BPTT | Gradients flow back through all T timesteps; surrogate gradient enables this |

**Next steps:**
- Try more epochs or a deeper architecture
- Change `NUM_TIMESTEPS` and observe the effect on accuracy
- Swap the surrogate gradient: `sl.IAFSqueeze(..., surrogate_grad_fn=sinabs.activation.Gaussian())`
- Explore the [sinabs documentation](https://sinabs.readthedocs.io) 